# RLS GRANT — обвязка DRP / Impala

SQL Lab режет `current_user` и не даёт SELECT на `rls_acq_user`.
Отсюда: кто ты в GP, GRANT, проверка чтения ACL.

Impala для GRANT не нужен — ячейка опциональная (тот же keytab, что в `final_script_2_1`).

Секция 5: INSERT нового пользователя в ACL (после логина DRP).


In [ ]:
import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)

NOTEBOOK_REV = '2026-09-25-rls-grant-v4'
drp_schema = 'sbx_da'
drp_superset_grant_role = 'raisa_superset'

ACL_TABLES = [
    'rls_acq_user',
    'rls_acq_user_filial',
    'rls_acq_role_sheet',
]

# Кого наделить SELECT. Смешанный регистр / дефис — в кавычках внутри GRANT.
grant_roles = [
    drp_superset_grant_role,
    'Shestopalov-VYur',
]

run_impala = False  # True — probe Impala тем же keytab

print('rev', NOTEBOOK_REV)
print('grant_roles', grant_roles)
print('tables', [f'{drp_schema}.{t}' for t in ACL_TABLES])


## 1) DRP


In [ ]:
def pg_ident(name):
    return '"' + str(name).replace('"', '""') + '"'


drp_user = input('DRP user: ').strip()
drp_password = getpass.getpass('DRP password: ')
drp = connect(
    to='DRP',
    user_params={'user_name': drp_user, 'password': drp_password},
)
print('rev', NOTEBOOK_REV, '| DRP login', drp_user)

with drp:
    who = drp.fetch(
        "SELECT current_user AS gp_user, session_user AS gp_session"
    )
    print('=== кто ты в Greenplum ===')
    display(who)

    try:
        drp.execute(
            'GRANT USAGE ON SCHEMA ' + drp_schema + ' TO ' + pg_ident(drp_superset_grant_role)
        )
        print('OK GRANT USAGE', drp_schema, drp_superset_grant_role)
    except Exception as exc:
        print('FAIL USAGE', type(exc).__name__, str(exc)[:300])

    for t in ACL_TABLES:
        fq = drp_schema + '.' + t
        for role in grant_roles:
            sql = 'GRANT SELECT ON TABLE ' + fq + ' TO ' + pg_ident(role)
            try:
                drp.execute(sql)
                print('OK', sql)
            except Exception as exc:
                print('FAIL', sql, type(exc).__name__, str(exc)[:300])

    for t in ACL_TABLES:
        fq = drp_schema + '.' + t
        n = drp.fetch('SELECT COUNT(*) AS n FROM ' + fq)
        print(fq, 'rows=', int(pd.to_numeric(n.iloc[0, 0], errors='coerce')))
    display(drp.fetch(
        'SELECT username, role, is_all_filials FROM '
        + drp_schema + '.rls_acq_user ORDER BY 1'
    ))

print('Дальше в SQL Lab — только vd_acq_rls_overview, без current_user.')


## 2) GRANT SELECT на ACL

Владелец таблиц = тот, кто гонял `rls_acq_roles_build`. Если GRANT упадёт — зайди тем же логином.


In [ ]:
print('SKIP: GRANT уже в ячейке «1) DRP». Если rev не v2 — закрой тетрадку и открой файл с диска.')


## 3) Проверка чтения


In [ ]:
print('SKIP: проверка уже в ячейке «1) DRP».')


## 4) Impala (по желанию)


In [ ]:
if not run_impala:
    print('SKIP Impala (run_impala=False)')
else:
    if 'imp' in globals() and imp is not None:
        print('Reuse Impala')
    else:
        imp = connect(
            to='IMPALA',
            extra_options={'db': 'sandbox_ai'},
            driver_args={'tez.queue.name': 'ai'},
            kerberos={
                'keytab_path': '/home/jovyan/test_requests/tech.keytab',
                'use_credentials': True,
                'update_keytab': True,
            },
            user_params={'user_name': 'Shestopalov-VYur'},
        )
        imp._init_connection()
        print('Impala connected')
    with imp:
        try:
            imp.execute('set MEM_LIMIT=4g')
        except Exception:
            pass
        probe = imp.fetch('SELECT 1 AS ok')
    display(probe)


## 5) Добавить пользователя ACL

Сначала отработай ячейку «1) DRP» (нужен `drp`). SQL Lab часто не даёт INSERT.

Логин = `SELECT '{{ current_username() }}'` под этим человеком в RAISA.

| Роль | Листы | `is_all_filials` |
|---|---|---|
| `GO_DTPP` | все | `1` |
| `GO_BIZ` | эффект. + клиенты | `1` |
| `RF_ROE` | эффект. + клиенты + терминалы | `0` + строки в `filials` |
| `RF_KM` | эффект. + клиенты | `0` + один РФ |

`filial_filter` как на дашборде: `Липецкий РФ`, `Санкт-Петербургский РФ`, `ЦРМБ`.

Секция 5: **`lovyagina-eev`**, роль `RF_KM`, филиал `Липецкий РФ`. Прогони только DRP и эту ячейку.

Не гоняй Run All в `rls_acq_roles_build` — он DROP-нет таблицы и сотрёт этого человека, если его нет в `USERS`.


In [ ]:
if 'drp' not in globals() or drp is None:
    raise RuntimeError('Сначала ячейка «1) DRP» — нужен объект drp.')

new_username = 'lovyagina-eev'      # RAISA, КМ Липецкий РФ
new_role = 'RF_KM'
is_all_filials = '0'
filials = ['Липецкий РФ']
comment = 'РФ КМ, Липецкий РФ'

assert new_role in ('GO_DTPP', 'GO_BIZ', 'RF_ROE', 'RF_KM')
if not str(new_username).strip():
    raise RuntimeError('Вставь new_username — логин из RAISA (SELECT current_username под КМ)')
if is_all_filials != '1' and not filials:
    raise RuntimeError('РФ: укажи filials как на дашборде')

u = new_username.replace("'", "''")
role = new_role.replace("'", "''")
cmt = comment.replace("'", "''")

sql_user = (
    "INSERT INTO sbx_da.rls_acq_user (username, role, is_all_filials, comment) "
    f"SELECT '{u}', '{role}', '{is_all_filials}', '{cmt}' "
    "WHERE NOT EXISTS ("
    "  SELECT 1 FROM sbx_da.rls_acq_user "
    f"  WHERE lower(BTRIM(username)) = lower('{u}')"
    ")"
)

sql_fil = []
if is_all_filials != '1':
    for f in filials:
        ff = str(f).strip().replace("'", "''")
        sql_fil.append(
            "INSERT INTO sbx_da.rls_acq_user_filial (username, filial_filter) "
            f"SELECT '{u}', '{ff}' "
            "WHERE NOT EXISTS ("
            "  SELECT 1 FROM sbx_da.rls_acq_user_filial "
            f"  WHERE lower(BTRIM(username)) = lower('{u}') "
            f"    AND BTRIM(filial_filter) = '{ff}'"
            ")"
        )

print('cell5: вход в with drp — если дальше тишина, висит reconnect, не INSERT')
with drp:
    print('cell5: сессия открыта')
    drp.execute("SET statement_timeout = '30s'")
    display(drp.fetch('SELECT 1 AS probe'))
    print('cell5: probe ok, INSERT user')
    drp.execute(sql_user)
    print('cell5: user done')
    for s in sql_fil:
        print('cell5: INSERT filial')
        drp.execute(s)
    print('cell5: filial done')
    display(drp.fetch(
        "SELECT username, role, is_all_filials, comment "
        "FROM sbx_da.rls_acq_user "
        f"WHERE lower(BTRIM(username)) = lower('{u}')"
    ))
    display(drp.fetch(
        "SELECT username, filial_filter "
        "FROM sbx_da.rls_acq_user_filial "
        f"WHERE lower(BTRIM(username)) = lower('{u}') "
        "ORDER BY 2"
    ))

print('OK. VD/чарты не трогай — подхватят сами.')
